# 1. [RI/2025/PRELIM/P2/Q2]
A company is developing a knowledge-based game that allows users to test their general knowledge through a five-question quiz. The system runs on a client-server model using <u>socket programming</u>. The server retrieves questions from a <u>SQLite</u> database, receives and validates user responses, calculates scores, and stores them. After the quiz, the server returns a sorted leaderboard to the client, displaying the top three performers. You are to implement core functionalities of this game system.


## Task 1 
Write a program that uses SQL to create a SQLite database named `quiz.db`. The database will be used to store questions from `questions.csv` and scores of quiz users. 

Create a table `question` with the following fields: 
- `id` (Integer, Primary Key) 
- `question_text` (Text) 
- `option_a` (Text) 
- `option_b` (Text), 
- `option_c` (Text), 
- `option_d` (Text) 
- `correct_option` (Text, storing 'A', 'B', 'C', or 'D') 
 
 
Create another table `scores` with the following fields: 
- `username` (Text, Primary Key) 
- `score` (Integer) 
The `scores` table will be populated later in Task 2.4. 
 
Write a program to read quiz questions from a provided text file, `questions.csv` and insert this data into the `questions` table in your `quiz.db` database. 
<div style="text-align: right">[4]</div>


In [1]:
import csv
import sqlite3

with open("./resources/questions.csv") as f:
    data = list(csv.reader(f))[1::]

question_create = """
CREATE TABLE question(
id INTEGER PRIMARY KEY AUTOINCREMENT,
question_text TEXT,
option_a TEXT,
option_b TEXT,
option_c TEXT,
option_d TEXT,
correct_option TEXT CHECK (correct_option IN ('A','B','C','D'))
)"""

scores_create = """
CREATE TABLE scores(
username TEXT PK,
score INTEGER
)"""

con = sqlite3.connect("./quiz.db")
c = con.cursor()
c.execute(question_create)
c.execute(scores_create)

question_query = """INSERT INTO question(question_text,option_a,option_b,option_c,option_d,correct_option) VALUES (?,?,?,?,?,?)"""
c.executemany(question_query,data)

con.commit()
con.close()


## Task 2 
Declare a Python class named Question that encapsulates the data for a single quiz question. 

It should have these attributes: 
- `question_text`, 
- `option_a`, 
- `option_b`, 
- `option_c`, 
- `option_d`, and 
- `correct_option`. 

Include a constructor to initialise these attributes. 
 
Create a method, `to_str()`, to return a single quiz question **as a string** and when printed, will be in this format: 

<center>
<img src="img/2025_ri_p2_q2_img_1.png" width="600" align="center"/>
</center>
 
Create a method, `check_answer(userchoice)`, that when given a `userchoice`, returns `True`, if user guessed the correct option for the question. If `userchoice` is incorrect or not one of the four valid options (A/B/C/D), will return `False`. 
<div style="text-align: right">[3]</div>


In [2]:
class Question:
    def __init__(self,question_text,option_a,option_b,option_c,option_d,correct_option):
        self.question_text = question_text
        self.option_a = option_a
        self.option_b = option_b
        self.option_c = option_c
        self.option_d = option_d
        self.correct_option = correct_option

    def to_str(self):
        return f"""{self.question_text}\nA. {self.option_a}\nB. {self.option_b}\nC. {self.option_c} \nD. {self.option_d}\nYour answer (A/B/C/D):"""
    
    def check_answer(self,userchoice):
        return userchoice == self.correct_option

## Task 3 
Write a function `generate_5_questions()` that will: 
- connect to the database, `quiz.db`, 
- generate 5 unique questions 
- returns a list of 5 `Question` objects. 

Test your function. Use the `to_str()` method to print the contents of the 5 `Question` objects. 
<div style="text-align: right">[4]</div>


In [3]:
import sqlite3
import random

def generate_5_questions():
    con = sqlite3.connect("quiz.db")
    c = con.cursor()
    c.execute("SELECT * FROM question")
    db_data = list(map(lambda x: list(x)[1::],c.fetchall()))
    unique_questions = random.sample(db_data,5)
    con.close()
    instances = list(map(lambda x: Question(*x),unique_questions))
    return instances

questions = generate_5_questions()
for i in questions:
    print(i.to_str())

What is the capital of France?
A. Berlin
B. Madrid
C. Paris 
D. Rome
Your answer (A/B/C/D):
What is the largest planet?
A. Earth
B. Mars
C. Jupiter 
D. Saturn
Your answer (A/B/C/D):
What is the chemical symbol for water?
A. H2O
B. O2
C. CO2 
D. NaCl
Your answer (A/B/C/D):
What color is the sky on a clear day?
A. Red
B. Blue
C. Green 
D. Yellow
Your answer (A/B/C/D):
What is the boiling point of water?
A. 90Â°C
B. 95Â°C
C. 100Â°C 
D. 105Â°C
Your answer (A/B/C/D):


## Task 4 
Declare a Python class `QuizSession` that is used by the server to keep track of each new quiz attempt made by a user. 

The QuizSession class contains the following attributes: 
- `username` 
- `score` (default to `0` as each quiz is a fresh attempt) 

Implement the `updateScoresDB()` method: 
- If this `username` does not exist in the `scores` table, insert a new `username` and his/her score, and returns a message: `"Your score has been updated on quiz.db"` 
- If this `username` exists, it would mean that the player has already played the quiz before. 
 - If the score attained in this `QuizSession` is higher than a previously attained score, update score and return a message: `"Congratulations, you have achieved a new high score"`, 
 - Else, return the message: `"You did not outperform your previous score."` 
 
You can assume that usernames are unique in the `scores` table. 
<div style="text-align: right">[5]</div>


In [4]:
import sqlite3

class QuizSession:
    def __init__(self,username):
        self.username = username
        self.score = 0

    def updateScoresDB(self):
        con = sqlite3.connect("quiz.db")
        c = con.cursor()
    
        c.execute(
            "SELECT score FROM scores WHERE username = ?",
            (self.username,)
        )
        row = c.fetchone()
    
        if row is None:
            c.execute(
                "INSERT INTO scores VALUES (?, ?)",
                (self.username, self.score)
            )
            con.commit()
            con.close()
            return "Your score has been updated on quiz.db."
    
        old_score = row[0]
    
        if self.score > old_score:
            c.execute(
                "UPDATE scores SET score = ? WHERE username = ?",
                (self.score, self.username)
            )
            con.commit()
            con.close()
            return "Congratulations, you have achieved a new high score."
    
        con.close()
        return "You did not outperform your previous score."

                


## Task 5 
Write a function, `getLeaderBoard()` which queries the `scores` table and retrieves the Top 3 players by score, and returns a formatted string, breaking ties by sorting 
`username` alphabetically. Your function should return **a formatted string** in this format when printed: 

```text
Top 3 players
===============
Kelly: 5pts
Malcolm: 5pts
Joy: 4pts
```

<div style="text-align: right">[4]</div>


In [5]:
import sqlite3

def getLeaderBoard():
    con = sqlite3.connect("quiz.db")
    c = con.cursor()
    c.execute("SELECT username,score FROM scores ORDER BY score DESC,username ASC")
    data = c.fetchall()
    con.close()
    string = "Top 3 players\n======="
    for i in data[0:3]:
        string += f"\n{i[0]}: {i[1]}"
    return string

## Task 6 
Referring to the code in `client.ipynb`, build an iterative socket server code that: 
1. Accept a new client connection 
2. Request for username from the client 
3. Create a new `QuizSession` object for the `username` 
4. Sends 5 questions (1 at a time) to the client and receives answers. 
5. Validates answers and updates score in the `QuizSession` object 
6. Report final score to client 
7. Updates the `scores` table in `quiz.db` using `updateScoreDB()` method 
8. Sends client the leaderboard information, using `getLeaderboard()`function. 
9. Close connection with client 

You are advised to make use of your code completed in Task 2.2 to 2.5 to complete the above tasks, when possible. 

Run both server program and client programs in separate Jupyter notebooks, and save the output of the client-server interaction below the code cells. 

<div style="text-align: right">[10]</div>

In [ ]:
from socket import socket

s = socket()

port = 12345
s.bind(('',port))

s.listen(5)
while True:
    c, addr = s.accept()

    c.send("Enter your name:".encode()) # Point 2: string is taken from client.ipynb
    name = c.recv(4096).decode()
    sess = QuizSession(name) # Point 3
    questions = generate_5_questions()
    for question in questions: # Point 4: For loop is used to send 1 question at a time
        c.send(f"\n{question.to_str()}".encode()) # Sending Question
        ans = c.recv(4096).decode() # Receiving Answer
        if question.check_answer(ans): # Validating Answer
            sess.score += 1 # Updating Answer if user answered correctly
    c.send(f"\nFinal Score: {sess.score}".encode()) # Point 6
    resp = (sess.updateScoresDB())
    c.send(resp.encode()) # Added since the updateScoreDB Method has proper return strings
    c.send(f"\n{getLeaderBoard()}".encode()) # Point 8

    c.close()

: 